# Firewall risk scoring demo

This notebook uses **synthetic** firewall events. It demonstrates preprocessing, fitting Isolation Forest, producing per-event heuristic scores and aggregating by source IP. There are no labeled attacks here, so there is no measured detection accuracy.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
import pandas as pd
from risk_pipeline import preprocess_data, train_model, analyze
sample = pd.read_csv(root / "data" / "sample_firewall_logs.csv")
print(f"{len(sample)} synthetic records across {sample['SOURCE_IP'].nunique()} IPs")
sample.head()

In [ ]:
processed = preprocess_data(sample)
processed[["SOURCE_IP", "DATE_TIME", "BYTE_RATIO", "THREAT_PERIOD", "THREAT_BYTES", "IS_DENY"]].head()

In [ ]:
model = train_model(processed, contamination=0.05)
events, users = analyze(sample, model)
print("Outlier events:", int(events["ANOMALY_BIN"].sum()))
users[["SOURCE_IP", "RISK_SCORE", "RISK_LEVEL", "ANOMALY_RATE", "NUM_EVENTS"]].head(10)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(users["RISK_SCORE"], bins=15)
axes[0].set(title="Per-IP mean heuristic risk", xlabel="Risk score", ylabel="IPs")
top = users.head(10).iloc[::-1]
axes[1].barh(top["SOURCE_IP"], top["RISK_SCORE"])
axes[1].set(title="Ten highest-scoring synthetic IPs", xlabel="Mean risk score")
plt.tight_layout()
plt.show()

## Interpretation

The sample generator deliberately inserts unusual events for `192.0.2.250`. Its high score demonstrates that the pipeline and dashboard respond to those synthetic records; it **does not** establish precision or recall on real malicious traffic. The fixed weights and thresholds are heuristic choices.